# Starting of with the saving the data in a test train folders

In [20]:
import glob
import os
import random
import shutil
import string

categories_dirs = ["./0.0.Normal", "./10.0.Possible glaucoma", "./13.Dragged Disc", "./27.Laser Spots"]

all_images_list = [image_path for category_dir in categories_dirs for image_path in glob.glob(category_dir + "/*")]

paths_train_images = []
paths_test_images = []

random.seed(50)

random.shuffle(all_images_list)

for directory in [os.path.join('./data/train'), os.path.join('./data/test')]:
    if not os.path.isdir(directory):
        os.makedirs(directory)


# training data in train folder testing in test folder also the 8- 20 split

In [21]:
for index, file_path in enumerate(all_images_list):
    target_dir = os.path.join('./data/train') if index < int(len(all_images_list) * 0.80) else os.path.join('./data/test')
    image_file_name = file_path.split('/')[-1]
    if index < int(len(all_images_list) * 0.80):
        paths_train_images.append(os.path.join(target_dir, image_file_name))
    else:
        paths_test_images.append(os.path.join(target_dir, image_file_name))
    shutil.copy(file_path, os.path.join(target_dir, image_file_name))

# Assigning the lables i have extracted the data from the assosiated folders and saved them by strippin the number and the extensaion jpg

In [22]:

imges_train = []
imges_test = []

for each in paths_train_images:
  imges_train.append(os.path.splitext(each)[0])
training_all_labels = []
for i in range(len(imges_train)):
  imges_train[i] = imges_train[i].rstrip(string.digits)
  training_all_labels.append(imges_train[i].split('/')[-1])

for each in paths_test_images:
  imges_test.append(os.path.splitext(each)[0])
testing_all_labels = []
for i in range(len(imges_test)):
  imges_test[i] = imges_test[i].rstrip(string.digits)
  testing_all_labels.append(imges_test[i].split('/')[-1])


# Made the rgb numpy array for both the test and train

In [23]:
import numpy as np
import matplotlib.image as mpimg

# For train images
training_set_balances = np.zeros((len(paths_train_images), 3))
for position in range(len(paths_train_images)):
  image_load = mpimg.imread(paths_train_images[position])
  mean_values = np.array(image_load).mean(axis=(0, 1))
  mean_red = mean_values[0]
  mean_green = mean_values[1]
  mean_blue = mean_values[2]
  sum_colors = (mean_red + mean_green + mean_blue)
  training_set_balances[position, 2] = mean_green / sum_colors
  training_set_balances[position, 1] = mean_blue / sum_colors
  training_set_balances[position, 0] = mean_red / sum_colors

# For test images
testing_set_balances = np.zeros((len(paths_test_images), 3))
for position in range(len(paths_test_images)):
  image_load = mpimg.imread(paths_test_images[position])
  mean_values = np.array(image_load).mean(axis=(0, 1))
  mean_red = mean_values[0]
  mean_green = mean_values[1]
  mean_blue = mean_values[2]
  sum_colors = (mean_red + mean_green + mean_blue)
  testing_set_balances[position, 2] = mean_green / sum_colors
  testing_set_balances[position, 1] = mean_blue / sum_colors
  testing_set_balances[position, 0] = mean_red / sum_colors

# Training of the k2 knn model for knneighbors

In [24]:
from sklearn import neighbors

k2_model_classifier = neighbors.KNeighborsClassifier(n_neighbors=1, weights='distance')
k2_model_classifier.fit(training_set_balances, training_all_labels)

KNeighborsClassifier(n_neighbors=1, weights='distance')

# showing the auccracy this varies based on the random shuffle

In [27]:
pred = k2_model_classifier.predict(testing_set_balances)
score = k2_model_classifier.score(testing_set_balances, testing_all_labels)
print ("The accuracy of the k2 Model : ", round(score,3))

The accuracy of the k2 Model :  0.706


# Confusion matrix for the normal label

In [29]:
TP = 0  # True Positive
TN = 0  # True Negative
FP = 0  # False Positive
FN = 0  # False Negative

for i in range(len(pred)):
    if pred[i] == 'normal' and testing_all_labels[i] == 'normal':
        TP += 1
    elif pred[i] == 'normal' and testing_all_labels[i] != 'normal':
        FP += 1
    elif pred[i] != 'normal' and testing_all_labels[i] != 'normal':
        TN += 1
    else:
        FN += 1
print("TP: ", TP)
print("FP: ", FP)
print("TN: ", TN)
print("FN: ", FN)
print()

acc = (TP + TN)/len(testing_all_labels)
print (f"Auccracy of normal : ", round(acc,3))

TP:  5
FP:  1
TN:  10
FN:  1

Auccracy of normal :  0.882


# Confusion matrix for the dragdisc label

In [30]:
TP = 0  # True Positive
TN = 0  # True Negative
FP = 0  # False Positive
FN = 0  # False Negative

for i in range(len(pred)):
    if pred[i] == 'dragdisc' and testing_all_labels[i] == 'dragdisc':
        TP += 1
    elif pred[i] == 'dragdisc' and testing_all_labels[i] != 'dragdisc':
        FP += 1
    elif pred[i] != 'dragdisc' and testing_all_labels[i] != 'dragdisc':
        TN += 1
    else:
        FN += 1
print("TP: ", TP)
print("FP: ", FP)
print("TN: ", TN)
print("FN: ", FN)

acc = (TP + TN)/len(testing_all_labels)
print (f"Auccracy of dragdisc : ", round(acc,3))

TP:  1
FP:  1
TN:  15
FN:  0
Auccracy of dragdisc :  0.941


# Confusion matrix for the laserspot label

In [31]:
TP = 0  # True Positive
TN = 0  # True Negative
FP = 0  # False Positive
FN = 0  # False Negative

for i in range(len(pred)):
    if pred[i] == 'laserspot' and testing_all_labels[i] == 'laserspot':
        TP += 1
    elif pred[i] == 'laserspot' and testing_all_labels[i] != 'laserspot':
        FP += 1
    elif pred[i] != 'laserspot' and testing_all_labels[i] != 'laserspot':
        TN += 1
    else:
        FN += 1
print("TP: ", TP)
print("FP: ", FP)
print("TN: ", TN)
print("FN: ", FN)

acc = (TP + TN)/len(testing_all_labels)
print (f"Auccracy of laserspot : ", round(acc, 3))

TP:  3
FP:  0
TN:  11
FN:  3
Auccracy of laserspot :  0.824


A machine learning application for image classification is the program's main focus. It starts by compiling image paths, each of which represents a distinct class of images, from specified folders.
By taking the digits and file extensions out of the filenames, labels for the images are extracted. The average colour ratios (red, green, and blue) of each image are determined by normaling the results by the total colour intensity and returns the results in a NumPy array.
The main function of the software is to train a K-Nearest Neighbours (KNN) classifier with two neighbours using the colour ratio characteristics of the training images in conjunction with their labels. The testing dataset is used to assess the accuracy of the model.
